# YelpChi и Amazon

In [6]:
import scipy.io as sio
import numpy as np

# 1. Проверим первые байты файла
with open('data/YelpChi/YelpChi.mat', 'rb') as f:
    header = f.read(8)
    print("Первые 8 байт:", header)
    if header[:4] == b'MATL':
        print("Похоже на MAT-файл старой версии (MATLAB 5)")
    elif header[:4] == b'\x89HDF':
        print("Это HDF5 (MAT v7.3)")
    else:
        print("Неизвестный формат, не MAT")

# 2. Попробуем разные способы загрузки
try:
    data = sio.loadmat('data/YelpChi/YelpChi.mat')
    print("loadmat успешен, ключи:", list(data.keys()))
except Exception as e:
    print("Ошибка loadmat:", e)

# 3. Альтернатива: попробуем h5py на всякий случай (если всё же HDF5)
try:
    import h5py
    with h5py.File('data/YelpChi.mat', 'r') as f:
        print("h5py прочитал, ключи:", list(f.keys()))
except Exception as e:
    print("h5py не сработал:", e)

# 4. Попробуем numpy load (не для MAT, но для других форматов)
try:
    data_np = np.load('data/YelpChi/YelpChi.mat')
    print("np.load сработал, ключи:", list(data_np.keys()))
except:
    pass

# 5. Проверим размер файла
import os
print("Размер файла:", os.path.getsize('data/YelpChi/YelpChi.mat'), "байт")

Первые 8 байт: b'MATLAB 5'
Похоже на MAT-файл старой версии (MATLAB 5)
loadmat успешен, ключи: ['__header__', '__version__', '__globals__', 'homo', 'net_rur', 'net_rtr', 'net_rsr', 'features', 'label']
h5py не сработал: Unable to synchronously open file (file signature not found)
Размер файла: 207676376 байт


In [8]:
# Полный код: удаление старых CSV, создание новых с префиксом YelpChi_

import scipy.io as sio
import pandas as pd
from scipy.sparse import coo_matrix
import os
import glob

# Пути
data_dir = 'data'
mat_file = os.path.join(data_dir, 'YelpChi.mat')
prefix = 'YelpChi'

# 1. Удаляем старые CSV-файлы, которые создавались без префикса (перечисляем конкретные имена)
old_patterns = [
    'net_rur_edges.csv', 'net_rtr_edges.csv', 'net_rsr_edges.csv', 'homo_edges.csv',
    'features.csv', 'label.csv', 'YelpChi_full.csv'  # старый full тоже удалим, создадим заново
]
deleted = []
for fname in old_patterns:
    full_path = os.path.join(data_dir, fname)
    if os.path.exists(full_path):
        os.remove(full_path)
        deleted.append(fname)
if deleted:
    print(f"Удалены старые файлы: {', '.join(deleted)}")
else:
    print("Старых файлов не найдено.")


Удалены старые файлы: net_rur_edges.csv, net_rtr_edges.csv, net_rsr_edges.csv, homo_edges.csv, features.csv, label.csv, YelpChi_full.csv


In [9]:

# 2. Загружаем .mat файл
data = sio.loadmat(mat_file)
print("Файл загружен. Ключи:", list(data.keys()))

# 3. Сохраняем разреженные графы с префиксом
for name in ['net_rur', 'net_rtr', 'net_rsr', 'homo']:
    if name in data:
        mat = data[name]
        coo = coo_matrix(mat)
        edges = pd.DataFrame({'row': coo.row, 'col': coo.col, 'value': coo.data})
        out_file = os.path.join(data_dir, f'{prefix}_{name}_edges.csv')
        edges.to_csv(out_file, index=False)
        print(f"Сохранён {out_file}: {len(edges)} рёбер")
    else:
        print(f"Внимание: {name} отсутствует в файле")

# 4. Сохраняем признаки с префиксом
if 'features' in data:
    feats = data['features']
    if hasattr(feats, 'toarray'):
        feats = feats.toarray()
    df_feats = pd.DataFrame(feats)
    out_file = os.path.join(data_dir, f'{prefix}_features.csv')
    df_feats.to_csv(out_file, index=False)
    print(f"Сохранён {out_file}: форма {feats.shape}")
else:
    print("'features' не найдены")

# 5. Сохраняем метки с префиксом
if 'label' in data:
    labels = data['label'].flatten()
    df_labels = pd.DataFrame(labels, columns=['label'])
    out_file = os.path.join(data_dir, f'{prefix}_label.csv')
    df_labels.to_csv(out_file, index=False)
    print(f"Сохранён {out_file}: {len(labels)} записей")
else:
    print("'label' не найдены")

# 6. Объединённый файл (признаки + метка) с префиксом
if 'features' in data and 'label' in data:
    X = data['features']
    if hasattr(X, 'toarray'):
        X = X.toarray()
    y = data['label'].flatten()
    full = pd.DataFrame(X)
    full['label'] = y
    out_file = os.path.join(data_dir, f'{prefix}_full.csv')
    full.to_csv(out_file, index=False)
    print(f"Сохранён {out_file} (признаки + метка): форма {full.shape}")
else:
    print("Не удалось создать объединённый файл.")

print("\nГотово! Все файлы сохранены в 'data/' с префиксом 'YelpChi_'.")

Файл загружен. Ключи: ['__header__', '__version__', '__globals__', 'homo', 'net_rur', 'net_rtr', 'net_rsr', 'features', 'label']
Сохранён data/YelpChi_net_rur_edges.csv: 98630 рёбер
Сохранён data/YelpChi_net_rtr_edges.csv: 1147232 рёбер
Сохранён data/YelpChi_net_rsr_edges.csv: 6805486 рёбер
Сохранён data/YelpChi_homo_edges.csv: 7693958 рёбер
Сохранён data/YelpChi_features.csv: форма (45954, 32)
Сохранён data/YelpChi_label.csv: 45954 записей
Сохранён data/YelpChi_full.csv (признаки + метка): форма (45954, 33)

Готово! Все файлы сохранены в 'data/' с префиксом 'YelpChi_'.


In [12]:
# Полный код для Amazon.mat: проверка структуры и создание CSV с префиксом Amazon_

import scipy.io as sio
import pandas as pd
from scipy.sparse import coo_matrix
import os

data_dir = 'data'
mat_file = os.path.join(data_dir, 'Amazon.mat')
prefix = 'Amazon'

# 1. Загружаем файл
print("Загрузка", mat_file)
data = sio.loadmat(mat_file)
print("Ключи в файле:", list(data.keys()))

# 2. Определяем, какие переменные есть (исключая служебные)
ignore_keys = ['__header__', '__version__', '__globals__']
actual_keys = [k for k in data.keys() if k not in ignore_keys]
print("Полезные переменные:", actual_keys)

# 3. Сохраняем разреженные графы (если они есть)
#    Обычно это net_upu, net_usu, net_uvu, homo
graph_keys = ['net_upu', 'net_usu', 'net_uvu', 'homo']
for key in graph_keys:
    if key in data:
        mat = data[key]
        coo = coo_matrix(mat)
        edges = pd.DataFrame({'row': coo.row, 'col': coo.col, 'value': coo.data})
        out_file = os.path.join(data_dir, f'{prefix}_{key}_edges.csv')
        edges.to_csv(out_file, index=False)
        print(f"Сохранён {out_file}: {len(edges)} рёбер")
    else:
        print(f"Переменная '{key}' не найдена, пропускаем")

# 4. Сохраняем признаки (скорее всего, называется 'features')
if 'features' in data:
    feats = data['features']
    if hasattr(feats, 'toarray'):
        feats = feats.toarray()
    df_feats = pd.DataFrame(feats)
    out_file = os.path.join(data_dir, f'{prefix}_features.csv')
    df_feats.to_csv(out_file, index=False)
    print(f"Сохранён {out_file}: форма {feats.shape}")
else:
    print("Переменная 'features' не найдена, пропускаем")

# 5. Сохраняем метки (скорее всего, 'label')
if 'label' in data:
    labels = data['label'].flatten()
    df_labels = pd.DataFrame(labels, columns=['label'])
    out_file = os.path.join(data_dir, f'{prefix}_label.csv')
    df_labels.to_csv(out_file, index=False)
    print(f"Сохранён {out_file}: {len(labels)} записей")
else:
    print("Переменная 'label' не найдена, пропускаем")

# 6. Если есть и признаки, и метки, создаём объединённый файл
if 'features' in data and 'label' in data:
    X = data['features']
    if hasattr(X, 'toarray'):
        X = X.toarray()
    y = data['label'].flatten()
    full = pd.DataFrame(X)
    full['label'] = y
    out_file = os.path.join(data_dir, f'{prefix}_full.csv')
    full.to_csv(out_file, index=False)
    print(f"Сохранён {out_file}: форма {full.shape}")
else:
    print("Не хватает features или label для создания объединённого файла")

print("\nГотово! Все доступные данные из Amazon.mat сохранены в папке 'data' с префиксом 'Amazon_'.")

Загрузка data/Amazon.mat
Ключи в файле: ['__header__', '__version__', '__globals__', 'homo', 'net_upu', 'net_usu', 'net_uvu', 'features', 'label']
Полезные переменные: ['homo', 'net_upu', 'net_usu', 'net_uvu', 'features', 'label']
Сохранён data/Amazon_net_upu_edges.csv: 351216 рёбер
Сохранён data/Amazon_net_usu_edges.csv: 7132958 рёбер
Сохранён data/Amazon_net_uvu_edges.csv: 2073474 рёбер
Сохранён data/Amazon_homo_edges.csv: 8796784 рёбер
Сохранён data/Amazon_features.csv: форма (11944, 25)
Сохранён data/Amazon_label.csv: 11944 записей
Сохранён data/Amazon_full.csv: форма (11944, 26)

Готово! Все доступные данные из Amazon.mat сохранены в папке 'data' с префиксом 'Amazon_'.


# Ellptic

In [13]:
# Скачивание и распаковка датасета Elliptic Bitcoin в папку data/Elliptic/

import os
import urllib.request
import zipfile

# Пути
data_dir = 'data'
dataset_dir = os.path.join(data_dir, 'Elliptic')
os.makedirs(dataset_dir, exist_ok=True)

# Базовый URL (из исходников PyG)
base_url = 'https://data.pyg.org/datasets/elliptic'

# Файлы, которые нужно скачать (имена csv, которые внутри zip)
csv_files = [
    'elliptic_txs_features.csv',
    'elliptic_txs_edgelist.csv',
    'elliptic_txs_classes.csv'
]

for csv_name in csv_files:
    zip_url = f'{base_url}/{csv_name}.zip'
    zip_path = os.path.join(dataset_dir, f'{csv_name}.zip')

    print(f'Скачиваем {zip_url} ...')
    urllib.request.urlretrieve(zip_url, zip_path)

    print(f'Распаковываем {zip_path} ...')
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(dataset_dir)

    # Удаляем zip-архив
    os.remove(zip_path)
    print(f'Сохранили {os.path.join(dataset_dir, csv_name)}')

print('\nГотово! Все три CSV-файла находятся в папке data/Elliptic/')

import pandas as pd

# Загружаем данные
features = pd.read_csv('data/Elliptic/elliptic_txs_features.csv', header=None)
classes = pd.read_csv('data/Elliptic/elliptic_txs_classes.csv')

# В features первые два столбца: txId и time_step, остальные 165 — признаки
features.columns = ['txId', 'time_step'] + [f'f{i}' for i in range(165)]

# Мэппим классы: licit -> 0, illicit -> 1, unknown -> -1 (или оставляем как есть)
class_map = {'licit': 0, 'illicit': 1, 'unknown': -1}
classes['label'] = classes['class'].map(class_map)

# Объединяем по txId
full = features.merge(classes[['txId', 'label']], on='txId', how='left')
full.to_csv('data/Elliptic/Elliptic_full.csv', index=False)
print("Создан data/Elliptic/Elliptic_full.csv")

Скачиваем https://data.pyg.org/datasets/elliptic/elliptic_txs_features.csv.zip ...
Распаковываем data/Elliptic/elliptic_txs_features.csv.zip ...
Сохранили data/Elliptic/elliptic_txs_features.csv
Скачиваем https://data.pyg.org/datasets/elliptic/elliptic_txs_edgelist.csv.zip ...
Распаковываем data/Elliptic/elliptic_txs_edgelist.csv.zip ...
Сохранили data/Elliptic/elliptic_txs_edgelist.csv
Скачиваем https://data.pyg.org/datasets/elliptic/elliptic_txs_classes.csv.zip ...
Распаковываем data/Elliptic/elliptic_txs_classes.csv.zip ...
Сохранили data/Elliptic/elliptic_txs_classes.csv

Готово! Все три CSV-файла находятся в папке data/Elliptic/
Создан data/Elliptic/Elliptic_full.csv


# TalkingData

In [4]:
import os
import zipfile
import pandas as pd

# 1. Целевая папка
target_dir = 'data/TalkingData'
os.makedirs(target_dir, exist_ok=True)

# 2. Ищем ZIP-файл
zip_path = 'data/TalkingData/talkingdata-adtracking-fraud-detection.zip'

# 3. Распаковываем
print(f"Распаковка {zip_path} -> {target_dir}")
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(target_dir)

# 4. Список распакованных CSV
extracted = [f for f in os.listdir(target_dir) if f.endswith('.csv')]
print("Распакованы файлы:", extracted)

# 5. (Опционально) Быстрый просмотр каждого файла (первые 5 строк)
for fname in extracted:
    path = os.path.join(target_dir, fname)
    df_sample = pd.read_csv(path, nrows=5)
    print(f"\n{fname} (sample):")
    print(df_sample.head())
    # Если хотите загрузить целиком, раскомментируйте:
    # var_name = fname.replace('.csv', '').replace('-', '_')
    # globals()[var_name] = pd.read_csv(path)

print("\nГотово! Все CSV-файлы находятся в 'data/TalkingData'.")

Распаковка data/TalkingData/talkingdata-adtracking-fraud-detection.zip -> data/TalkingData
Распакованы файлы: ['test.csv', 'train_sample.csv', 'test_supplement.csv', 'train.csv', 'sample_submission.csv']

test.csv (sample):
   click_id      ip  app  device  os  channel           click_time
0         0    5744    9       1   3      107  2017-11-10 04:00:00
1         1  119901    9       1   3      466  2017-11-10 04:00:00
2         2   72287   21       1  19      128  2017-11-10 04:00:00
3         3   78477   15       1  13      111  2017-11-10 04:00:00
4         4  123080   12       1  13      328  2017-11-10 04:00:00

train_sample.csv (sample):
       ip  app  device  os  channel           click_time  attributed_time  \
0   87540   12       1  13      497  2017-11-07 09:30:38              NaN   
1  105560   25       1  17      259  2017-11-07 13:40:27              NaN   
2  101424   12       1  19      212  2017-11-07 18:05:24              NaN   
3   94584   13       1  13      477  2

# IEEE-CIS

In [2]:
import os
import zipfile
import pandas as pd

# 1. Настраиваем пути
target_dir = 'data/IEEE-CIS'
os.makedirs(target_dir, exist_ok=True)

zip_path = 'data/IEEE-CIS/ieee-fraud-detection.zip'

if zip_path is None:
    raise FileNotFoundError("Не удалось найти ieee-fraud-detection.zip. Укажите путь вручную.")

# 3. Распаковываем
print(f"Распаковка {zip_path} -> {target_dir}")
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(target_dir)

# 4. Выводим список распакованных файлов
extracted = [f for f in os.listdir(target_dir) if f.endswith('.csv')]
print("Распакованы CSV-файлы:", extracted)

# 5. (Опционально) Загружаем их в переменные Dataframe для быстрого взгляда
for fname in extracted:
    path = os.path.join(target_dir, fname)
    var_name = fname.replace('.csv', '').replace('-', '_')
    # Читаем первые 5 строк, чтобы не грузить огромные таблицы без надобности
    df = pd.read_csv(path, nrows=5)
    print(f"\n{var_name} (sample):\n", df.head())
    # Если хотите загрузить целиком в переменную, раскомментируйте:
    # globals()[var_name] = pd.read_csv(path)

Распаковка data/IEEE-CIS/ieee-fraud-detection.zip -> data/IEEE-CIS
Распакованы CSV-файлы: ['test_transaction.csv', 'train_identity.csv', 'test_identity.csv', 'sample_submission.csv', 'train_transaction.csv']

test_transaction (sample):
    TransactionID  TransactionDT  TransactionAmt ProductCD  card1  card2  \
0        3663549       18403224           31.95         W  10409  111.0   
1        3663550       18403263           49.00         W   4272  111.0   
2        3663551       18403310          171.00         W   4476  574.0   
3        3663552       18403310          284.95         W  10989  360.0   
4        3663553       18403317           67.95         W  18018  452.0   

   card3       card4  card5  card6  ...  V330  V331  V332  V333 V334  V335  \
0  150.0        visa  226.0  debit  ...   NaN   NaN   NaN   NaN  NaN   NaN   
1  150.0        visa  226.0  debit  ...   NaN   NaN   NaN   NaN  NaN   NaN   
2  150.0        visa  226.0  debit  ...   NaN   NaN   NaN   NaN  NaN   NaN   


# MGTAB

In [9]:
import os
import zipfile
import torch
import pandas as pd
import numpy as np
import subprocess
import sys
import shutil

# ------------------------------------------------------------
# Функции для конвертации .pt в CSV
# ------------------------------------------------------------
def convert_pt_to_csv(pt_path, output_dir):
    data = torch.load(pt_path, map_location='cpu')
    base_name = os.path.splitext(os.path.basename(pt_path))[0]
    if isinstance(data, dict):
        dict_dir = os.path.join(output_dir, base_name)
        os.makedirs(dict_dir, exist_ok=True)
        for key, value in data.items():
            save_path = os.path.join(dict_dir, f"{key}.csv")
            _save_tensor_to_csv(value, save_path)
        print(f"Словарь из {len(data)} ключей сохранён в {dict_dir}")
    else:
        save_path = os.path.join(output_dir, f"{base_name}.csv")
        _save_tensor_to_csv(data, save_path)

def _save_tensor_to_csv(data, save_path):
    if isinstance(data, torch.Tensor):
        if data.dim() == 0:
            df = pd.DataFrame([data.item()])
        else:
            arr = data.numpy()
            if arr.ndim == 1:
                df = pd.DataFrame(arr.reshape(-1, 1))
            elif arr.ndim == 2:
                df = pd.DataFrame(arr)
            else:
                new_shape = (arr.shape[0], -1)
                arr_2d = arr.reshape(new_shape)
                df = pd.DataFrame(arr_2d)
    elif isinstance(data, (list, np.ndarray)):
        if isinstance(data, list) and all(isinstance(x, (int, float)) for x in data):
            df = pd.DataFrame(data, columns=['value'])
        else:
            df = pd.DataFrame(data)
    elif isinstance(data, (int, float, str)):
        df = pd.DataFrame([data])
    else:
        print(f"Неизвестный тип {type(data)} для файла {save_path}, сохраняю строковое представление")
        df = pd.DataFrame([str(data)])
    df.to_csv(save_path, index=False)
    print(f"Сохранён {save_path}")

# ------------------------------------------------------------
# Функция распаковки с поддержкой разных методов сжатия
# ------------------------------------------------------------
def extract_archive(archive_path, target_dir):
    try:
        with zipfile.ZipFile(archive_path, 'r') as zf:
            zf.extractall(target_dir)
        print(f"Успешно распаковано через zipfile: {archive_path}")
        return
    except NotImplementedError:
        print(f"Стандартный zipfile не поддерживает метод сжатия. Пробуем patool...")
        try:
            import patoolib
        except ImportError:
            print("patool не установлен. Устанавливаем...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "patool"])
            import patoolib
        patoolib.extract_archive(archive_path, outdir=target_dir, interactive=False)
        print(f"Распаковано через patool: {archive_path}")
        return
    except Exception as e:
        raise RuntimeError(f"Не удалось распаковать архив {archive_path}: {e}")

# ------------------------------------------------------------
# Основная функция обработки ZIP -> CSV
# ------------------------------------------------------------
def process_zip_archive(zip_name, target_parent_dir):
    # Поиск архива (сначала как есть, потом в подпапках)
    zip_path = None
    if os.path.exists(zip_name):
        zip_path = zip_name
    else:
        # Ищем в текущей папке и в data/
        for cand in [zip_name, f'./{zip_name}', f'data/{zip_name}', f'../{zip_name}', f'/content/{zip_name}']:
            if os.path.exists(cand):
                zip_path = cand
                break
    if zip_path is None:
        raise FileNotFoundError(f"Не найден архив {zip_name}")

    base_name = os.path.splitext(os.path.basename(zip_name))[0]
    output_root = os.path.join(target_parent_dir, base_name)
    os.makedirs(output_root, exist_ok=True)

    extract_dir = os.path.join(output_root, '_temp_extract')
    os.makedirs(extract_dir, exist_ok=True)

    extract_archive(zip_path, extract_dir)

    # Поиск всех .pt файлов
    pt_files = []
    for root, dirs, files in os.walk(extract_dir):
        for file in files:
            if file.endswith('.pt'):
                pt_files.append(os.path.join(root, file))

    print(f"Найдено {len(pt_files)} .pt файлов в {extract_dir}")

    for pt_file in pt_files:
        rel_path = os.path.relpath(pt_file, extract_dir)
        out_dir = os.path.join(output_root, os.path.dirname(rel_path))
        os.makedirs(out_dir, exist_ok=True)
        convert_pt_to_csv(pt_file, out_dir)

    # Удаляем временную папку
    shutil.rmtree(extract_dir)
    print(f"Обработка {base_name} завершена. Результаты в {output_root}")

# ------------------------------------------------------------
# Запуск
# ------------------------------------------------------------
data_dir = 'data/MGTAB/'
os.makedirs(data_dir, exist_ok=True)

# Обрабатываем MGTAB.zip (если есть)
try:
    process_zip_archive('data/MGTAB.zip', data_dir)
except FileNotFoundError as e:
    print(e)
    print("Пропускаем MGTAB.zip, его нет.")


Успешно распаковано через zipfile: data/MGTAB.zip
Найдено 6 .pt файлов в data/MGTAB/MGTAB/_temp_extract
Сохранён data/MGTAB/MGTAB/MGTAB/edge_type.csv
Сохранён data/MGTAB/MGTAB/MGTAB/labels_stance.csv
Сохранён data/MGTAB/MGTAB/MGTAB/labels_bot.csv
Сохранён data/MGTAB/MGTAB/MGTAB/edge_weight.csv
Сохранён data/MGTAB/MGTAB/MGTAB/edge_index.csv
Сохранён data/MGTAB/MGTAB/MGTAB/features.csv
Обработка MGTAB завершена. Результаты в data/MGTAB/MGTAB


In [ ]:
import pandas as pd
import json

# Загружаем JSON
json_path = 'data/your_file.json'   # укажите путь
csv_path = 'data/your_file.csv'     # куда сохранить

# Чтение JSON (поддерживаются строки, список словарей, вложенные структуры)
with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# Если JSON — список словарей, то прямо в DataFrame
if isinstance(data, list):
    df = pd.DataFrame(data)
else:
    # Если JSON — один словарь, оборачиваем в список
    df = pd.DataFrame([data])

# Сохраняем в CSV
df.to_csv(csv_path, index=False, encoding='utf-8')
print(f"Сохранён {csv_path}, форма: {df.shape}")

# Social Honeypot Dataset

In [18]:
import csv
import os

# Папка для выходных файлов
output_dir = "data/SHD"
os.makedirs(output_dir, exist_ok=True)   # создаём папку, если её нет

# Список файлов: (входной файл, выходной CSV, имена столбцов)
files = [
    ("data/content_polluters.txt", "content_polluters.csv",
     ["UserID", "CreatedAt", "CollectedAt", "NumberOfFollowings",
      "NumberOfFollowers", "NumberOfTweets", "LengthOfScreenName",
      "LengthOfDescriptionInUserProfile"]),
    ("data/content_polluters_followings.txt", "content_polluters_followings.csv",
     ["UserID", "SeriesOfNumberOfFollowings"]),
    ("data/content_polluters_tweets.txt", "content_polluters_tweets.csv",
     ["UserID", "TweetID", "Tweet", "CreatedAt"]),
    ("data/legitimate_users.txt", "legitimate_users.csv",
     ["UserID", "CreatedAt", "CollectedAt", "NumberOfFollowings",
      "NumberOfFollowers", "NumberOfTweets", "LengthOfScreenName",
      "LengthOfDescriptionInUserProfile"]),
    ("data/legitimate_users_followings.txt", "legitimate_users_followings.csv",
     ["UserID", "SeriesOfNumberOfFollowings"]),
    ("data/legitimate_users_tweets.txt", "legitimate_users_tweets.csv",
     ["UserID", "TweetID", "Tweet", "CreatedAt"])
]

for infile, outfile, fieldnames in files:
    if not os.path.exists(infile):
        print(f"Файл {infile} не найден, пропускаем")
        continue

    # Формируем полный путь к выходному файлу внутри папки SHD
    outpath = os.path.join(output_dir, outfile)

    with open(infile, 'r', encoding='utf-8') as fin, \
         open(outpath, 'w', encoding='utf-8', newline='') as fout:

        reader = csv.reader(fin, delimiter='\t')
        writer = csv.DictWriter(fout, fieldnames=fieldnames, quoting=csv.QUOTE_MINIMAL)

        writer.writeheader()
        for row in reader:
            # Если в строке меньше полей, чем ожидалось, дополняем пустыми значениями
            if len(row) < len(fieldnames):
                row += [''] * (len(fieldnames) - len(row))
            # Создаём словарь для записи
            d = dict(zip(fieldnames, row))
            writer.writerow(d)

    print(f"✅ {infile} -> {outpath}")

✅ data/content_polluters.txt -> data/SHD/content_polluters.csv
✅ data/content_polluters_followings.txt -> data/SHD/content_polluters_followings.csv
✅ data/content_polluters_tweets.txt -> data/SHD/content_polluters_tweets.csv
✅ data/legitimate_users.txt -> data/SHD/legitimate_users.csv
✅ data/legitimate_users_followings.txt -> data/SHD/legitimate_users_followings.csv
✅ data/legitimate_users_tweets.txt -> data/SHD/legitimate_users_tweets.csv


# Cresci-2017 (Social Spambots Detection)

# midterm и pronbots

In [20]:
import pandas as pd
import json

# Загружаем JSON
json_path = 'data/midterm-2018/midterm-2018_processed_user_objects.json'   # укажите путь
csv_path = 'data/midterm-2018/midterm-2018_processed_user_objects.csv'     # куда сохранить

# Чтение JSON (поддерживаются строки, список словарей, вложенные структуры)
with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# Если JSON — список словарей, то прямо в DataFrame
if isinstance(data, list):
    df = pd.DataFrame(data)
else:
    # Если JSON — один словарь, оборачиваем в список
    df = pd.DataFrame([data])

# Сохраняем в CSV
df.to_csv(csv_path, index=False, encoding='utf-8')
print(f"Сохранён {csv_path}, форма: {df.shape}")

Сохранён data/midterm-2018/midterm-2018_processed_user_objects.csv, форма: (50538, 19)


In [23]:
import pandas as pd
import json
from pandas import json_normalize

# Пути
json_path = 'data/pronbots-2019/pronbots-2019_tweets.json'
csv_path = 'data/pronbots-2019/pronbots-2019_tweets.csv'

# 1. Загружаем исходный JSON
with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# Преобразуем в DataFrame
if isinstance(data, list):
    df = pd.DataFrame(data)
else:
    df = pd.DataFrame([data])

print(f"Исходная форма: {df.shape}")
print("Столбцы:", list(df.columns))

# 2. Проверяем, есть ли столбец 'C2'
if 'user' in df.columns:
    # Применяем json_normalize ко всем значениям в столбце C2
    # Предполагаем, что каждый элемент — либо dict, либо JSON-строка
    def parse_c2(val):
        if isinstance(val, str):
            try:
                return json.loads(val)
            except:
                return {}
        elif isinstance(val, dict):
            return val
        else:
            return {}

    parsed = df['user'].apply(parse_c2)

    # Нормализуем распарсенные словари в DataFrame
    # Если в C2 хранятся однородные объекты, можно использовать json_normalize(parsed)
    # Но apply возвращает Series из dict, поэтому json_normalize(parsed) работает
    expanded = pd.json_normalize(parsed)   # создаёт отдельные столбцы для каждого ключа внутри C2

    # Удаляем исходный столбец C2 и объединяем с новыми столбцами
    df = df.drop(columns=['user']).reset_index(drop=True)
    df = pd.concat([df, expanded], axis=1)

    print(f"После раскрытия C2: {df.shape}")
    print("Новые столбцы:", list(expanded.columns))
else:
    print("Столбец 'C2' не найден, сохранение без раскрытия.")

# 3. Сохраняем результат в CSV
df.to_csv(csv_path, index=False, encoding='utf-8')
print(f"\nСохранён: {csv_path}")

Исходная форма: (17882, 2)
Столбцы: ['created_at', 'user']
После раскрытия C2: (17882, 44)
Новые столбцы: ['follow_request_sent', 'has_extended_profile', 'profile_use_background_image', 'default_profile_image', 'id', 'profile_background_image_url_https', 'verified', 'translator_type', 'profile_text_color', 'profile_image_url_https', 'profile_sidebar_fill_color', 'followers_count', 'profile_sidebar_border_color', 'id_str', 'profile_background_color', 'listed_count', 'is_translation_enabled', 'utc_offset', 'statuses_count', 'description', 'friends_count', 'location', 'profile_link_color', 'profile_image_url', 'following', 'geo_enabled', 'profile_banner_url', 'profile_background_image_url', 'screen_name', 'lang', 'profile_background_tile', 'favourites_count', 'name', 'notifications', 'url', 'created_at', 'contributors_enabled', 'time_zone', 'protected', 'default_profile', 'is_translator', 'entities.description.urls', 'entities.url.urls']

Сохранён: data/pronbots-2019/pronbots-2019_tweets.

# Считывание всех файлов для удобного понимания их структуры

In [25]:
import os
import pandas as pd

# Настройка pandas для отображения всех столбцов и строк без сокращений
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
pd.set_option('display.max_rows', 5)  # для head(5) этого достаточно

# Путь к папке data (убедитесь, что ноутбук запущен в нужной директории)
data_dir = './data'

if not os.path.exists(data_dir):
    print(f"Папка '{data_dir}' не найдена. Укажите правильный путь.")
else:
    # Рекурсивный обход всех подпапок
    for root, dirs, files in os.walk(data_dir):
        for file in files:
            if file.endswith('.csv'):
                full_path = os.path.join(root, file)
                # Определяем относительный путь от data_dir (папка-владелец)
                rel_path = os.path.relpath(root, data_dir)
                if rel_path == '.':
                    owner = 'Корневая папка data'
                else:
                    owner = rel_path

                # Вывод информации о файле
                print("\n" + "="*80)
                print(f"📁 Папка: {owner}")
                print(f"📄 Файл: {file}")
                print("="*80)

                try:
                    df = pd.read_csv(full_path)
                    print(f"✅ Размер: {df.shape[0]} строк × {df.shape[1]} столбцов")
                    print("\nПервые 5 строк (все столбцы):\n")
                    print(df.head(5).to_string(index=False))
                except Exception as e:
                    print(f"⚠️ Ошибка при чтении файла: {e}")

                print("\n" + "🔹" * 40 + "\n")  # разделитель между датасетами


📁 Папка: Корневая папка data
📄 Файл: creditcard.csv
✅ Размер: 284807 строк × 31 столбцов

Первые 5 строк (все столбцы):

 Time        V1        V2       V3        V4        V5        V6        V7        V8        V9       V10       V11       V12       V13       V14       V15       V16       V17       V18       V19       V20       V21       V22       V23       V24       V25       V26       V27       V28  Amount  Class
  0.0 -1.359807 -0.072781 2.536347  1.378155 -0.338321  0.462388  0.239599  0.098698  0.363787  0.090794 -0.551600 -0.617801 -0.991390 -0.311169  1.468177 -0.470401  0.207971  0.025791  0.403993  0.251412 -0.018307  0.277838 -0.110474  0.066928  0.128539 -0.189115  0.133558 -0.021053  149.62      0
  0.0  1.191857  0.266151 0.166480  0.448154  0.060018 -0.082361 -0.078803  0.085102 -0.255425 -0.166974  1.612727  1.065235  0.489095 -0.143772  0.635558  0.463917 -0.114805 -0.183361 -0.145783 -0.069083 -0.225775 -0.638672  0.101288 -0.339846  0.167170  0.125895 -0.008983  0.

/var/folders/md/78mwsg9n23525_j8040kg6fw0000gn/T/ipykernel_34509/298777058.py:35: DtypeWarning: Columns (1,8,9,10,11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(full_path)


✅ Размер: 18790469 строк × 7 столбцов

Первые 5 строк (все столбцы):

 click_id     ip  app  device  os  channel          click_time
        0   5744    9       1   3      107 2017-11-10 04:00:00
        1 119901    9       1   3      466 2017-11-10 04:00:00
        2  72287   21       1  19      128 2017-11-10 04:00:00
        3  78477   15       1  13      111 2017-11-10 04:00:00
        4 123080   12       1  13      328 2017-11-10 04:00:00

🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹


📁 Папка: TalkingData
📄 Файл: train_sample.csv
✅ Размер: 100000 строк × 8 столбцов

Первые 5 строк (все столбцы):

    ip  app  device  os  channel          click_time attributed_time  is_attributed
 87540   12       1  13      497 2017-11-07 09:30:38             NaN              0
105560   25       1  17      259 2017-11-07 13:40:27             NaN              0
101424   12       1  19      212 2017-11-07 18:05:24             NaN              0
 94584   13       1  13      477 2017-11-07 04:58:08      

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



✅ Размер: 10199 строк × 788 столбцов

Первые 5 строк (все столбцы):

  0   1   2        3   4        5        6        7   8        9       10       11       12       13       14  15  16  17  18  19       20       21       22       23        24       25        26        27        28        29        30        31       32       33        34        35        36        37        38       39       40       41        42        43        44       45        46       47        48        49        50        51        52       53        54       55       56        57       58        59        60       61        62        63        64       65       66        67       68       69       70        71       72        73        74       75        76       77        78        79        80        81       82        83       84        85        86       87        88        89        90        91        92       93       94       95        96       97       98        99      100      101       102      1

# Предобработка данных

In [31]:
import os
import shutil
import pandas as pd
import numpy as np

# ==================== КОНФИГУРАЦИЯ ====================
SOURCE_ROOT = "data"          # исходная папка с датасетами
TARGET_ROOT = "final_data"    # новая папка для обработанных датасетов

# Список датасетов и их тип обработки
# 'copy_file'       : просто копируем один файл
# 'copy_folder'     : копируем всю папку (для графовых)
# 'merge_cresci'    : специальная обработка Cresci-2017
# 'merge_midterm'   : специальная обработка Midterm-2018

datasets_to_prepare = {
    "Credit Card Fraud": {
        "type": "copy_file",
        "source": os.path.join(SOURCE_ROOT, "creditcard.csv"),
        "target": os.path.join(TARGET_ROOT, "creditcard.csv")
    },
    "IEEE-CIS": {
        "type": "copy_file",
        "source": os.path.join(SOURCE_ROOT, "IEEE-CIS", "train_transaction.csv"),
        "target": os.path.join(TARGET_ROOT, "ieee_train_transaction.csv")
    },
    "PaySim": {
        "type": "copy_file",
        "source": os.path.join(SOURCE_ROOT, "PaySim.csv"),
        "target": os.path.join(TARGET_ROOT, "paysim.csv")
    },
    "AIZBank": {
        "type": "copy_file",
        "source": os.path.join(SOURCE_ROOT, "AIZBank-full.csv"),
        "target": os.path.join(TARGET_ROOT, "aizbank.csv")
    },
    "Elliptic (Bitcoin)": {
        "type": "copy_folder",
        "source": os.path.join(SOURCE_ROOT, "Elliptic"),
        "target": os.path.join(TARGET_ROOT, "Elliptic")
    },
    "Amazon Reviews": {
        "type": "copy_folder",
        "source": os.path.join(SOURCE_ROOT, "Amazon"),
        "target": os.path.join(TARGET_ROOT, "Amazon")
    },
    "Yelp Reviews": {
        "type": "copy_folder",
        "source": os.path.join(SOURCE_ROOT, "YelpChi"),
        "target": os.path.join(TARGET_ROOT, "YelpChi")
    },
    "Click Fraud": {
        "type": "copy_file",
        "source": os.path.join(SOURCE_ROOT, "TalkingData", "train_sample.csv"),
        "target": os.path.join(TARGET_ROOT, "click_fraud_sample.csv")
    },
    "Social Honeypot": {
        "type": "copy_file",  # объединённый файл мы не создаём, просто копируем оба оригинальных?
        # Но в экспериментах удобнее один файл. Создадим объединённый CSV.
        "type": "merge_social_honeypot",  # создадим отдельную обработку
        "source_cp": os.path.join(SOURCE_ROOT, "SHD", "content_polluters.csv"),
        "source_legit": os.path.join(SOURCE_ROOT, "SHD", "legitimate_users.csv"),
        "target": os.path.join(TARGET_ROOT, "social_honeypot.csv")
    },
    "MGTAB": {
        "type": "copy_folder",
        "source": os.path.join(SOURCE_ROOT, "MGTAB"),
        "target": os.path.join(TARGET_ROOT, "MGTAB")
    },
    "Cresci-2017": {
        "type": "merge_cresci",
        "genuine_path": os.path.join(SOURCE_ROOT, "Cresci-2017", "genuine_accounts.csv", "users.csv"),
        "spambot_folders": [
            "traditional_spambots_1.csv",
            "traditional_spambots_2.csv",
            "traditional_spambots_3.csv",
            "traditional_spambots_4.csv"
        ],
        "target": os.path.join(TARGET_ROOT, "cresci_merged.csv")
    },
    "Midterm-2018": {
        "type": "merge_midterm",
        "features_path": os.path.join(SOURCE_ROOT, "midterm-2018", "midterm-2018_processed_user_objects.csv"),
        "labels_path": os.path.join(SOURCE_ROOT, "midterm-2018", "midterm-2018.tsv"),
        "target": os.path.join(TARGET_ROOT, "midterm_merged.csv")
    },
    "Pronbots-2019": {
        "type": "copy_file",
        "source": os.path.join(SOURCE_ROOT, "pronbots-2019", "pronbots-2019_tweets.csv"),
        "target": os.path.join(TARGET_ROOT, "pronbots_tweets.csv")
    }
}

# ==================== ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ====================

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def copy_file(src, dst):
    ensure_dir(os.path.dirname(dst))
    shutil.copy2(src, dst)
    print(f"  Скопирован: {src} -> {dst}")

def copy_folder(src, dst):
    ensure_dir(dst)
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f"  Скопирована папка: {src} -> {dst}")

def merge_cresci(genuine_path, spambot_folders, target):
    """Объединяет genuine и traditional_spambots, оставляет числовые колонки + label"""
    all_dfs = []

    # Genuine
    try:
        genuine = pd.read_csv(genuine_path, encoding='utf-8', low_memory=False)
    except:
        genuine = pd.read_csv(genuine_path, encoding='latin1', low_memory=False)
    genuine['label'] = 0
    all_dfs.append(genuine)
    print(f"    Загружено genuine: {len(genuine)} записей")

    # Spambots
    for folder in spambot_folders:
        spambot_path = os.path.join(SOURCE_ROOT, "Cresci-2017", folder, "users.csv")
        if os.path.exists(spambot_path):
            try:
                spambot = pd.read_csv(spambot_path, encoding='utf-8', low_memory=False)
            except:
                spambot = pd.read_csv(spambot_path, encoding='latin1', low_memory=False)
            spambot['label'] = 1
            all_dfs.append(spambot)
            print(f"    Загружено {folder}: {len(spambot)} записей")

    df = pd.concat(all_dfs, ignore_index=True)
    # Оставляем только числовые колонки + label
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if 'label' not in numeric_cols:
        numeric_cols.append('label')
    df = df[numeric_cols]

    ensure_dir(os.path.dirname(target))
    df.to_csv(target, index=False)
    print(f"    Сохранён объединённый Cresci-2017: {df.shape} -> {target}")

def merge_midterm(features_path, labels_path, target):
    """Объединяет CSV признаков и TSV меток по user_id"""
    # Загружаем признаки (engine='python' для обработки проблемных строк)
    df_features = pd.read_csv(features_path, engine='python', encoding='utf-8', on_bad_lines='skip')
    print(f"    Загружено признаков: {len(df_features)} строк")

    # Загружаем метки
    df_labels = pd.read_csv(labels_path, sep='\t', header=None, names=['user_id', 'label_raw'])
    df_labels['label'] = df_labels['label_raw'].map({'bot': 1, 'human': 0})
    df_labels = df_labels.drop(columns=['label_raw'])
    print(f"    Загружено меток: {len(df_labels)} строк")

    # Объединяем
    df_features['user_id'] = df_features['user_id'].astype(str)
    df_labels['user_id'] = df_labels['user_id'].astype(str)
    df = df_features.merge(df_labels, on='user_id', how='inner')
    print(f"    После объединения: {len(df)} строк")

    ensure_dir(os.path.dirname(target))
    df.to_csv(target, index=False)
    print(f"    Сохранён объединённый Midterm-2018: {df.shape} -> {target}")

def merge_social_honeypot(cp_path, legit_path, target):
    """Объединяет content_polluters (label=1) и legitimate_users (label=0)"""
    cp = pd.read_csv(cp_path)
    legit = pd.read_csv(legit_path)
    cp['label'] = 1
    legit['label'] = 0
    df = pd.concat([cp, legit], ignore_index=True)
    # Оставляем только числовые колонки + label
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if 'label' not in numeric_cols:
        numeric_cols.append('label')
    df = df[numeric_cols]
    ensure_dir(os.path.dirname(target))
    df.to_csv(target, index=False)
    print(f"    Сохранён Social Honeypot: {df.shape} -> {target}")

# ==================== ОСНОВНОЙ ПРОЦЕСС ====================

def main():
    print("Создание папки final_data и подготовка датасетов...")
    ensure_dir(TARGET_ROOT)

    for name, cfg in datasets_to_prepare.items():
        print(f"\nОбработка: {name}")
        if cfg["type"] == "copy_file":
            copy_file(cfg["source"], cfg["target"])
        elif cfg["type"] == "copy_folder":
            copy_folder(cfg["source"], cfg["target"])
        elif cfg["type"] == "merge_cresci":
            merge_cresci(cfg["genuine_path"], cfg["spambot_folders"], cfg["target"])
        elif cfg["type"] == "merge_midterm":
            merge_midterm(cfg["features_path"], cfg["labels_path"], cfg["target"])
        elif cfg["type"] == "merge_social_honeypot":
            merge_social_honeypot(cfg["source_cp"], cfg["source_legit"], cfg["target"])
        else:
            print(f"  Неизвестный тип: {cfg['type']}")

    print("\n" + "="*60)
    print("Готово! Все датасеты сохранены в папке 'final_data'.")
    print("="*60)

if __name__ == "__main__":
    main()

Создание папки final_data и подготовка датасетов...

Обработка: Credit Card Fraud
  Скопирован: data/creditcard.csv -> final_data/creditcard.csv

Обработка: IEEE-CIS
  Скопирован: data/IEEE-CIS/train_transaction.csv -> final_data/ieee_train_transaction.csv

Обработка: PaySim
  Скопирован: data/PaySim.csv -> final_data/paysim.csv

Обработка: AIZBank
  Скопирован: data/AIZBank-full.csv -> final_data/aizbank.csv

Обработка: Elliptic (Bitcoin)
  Скопирована папка: data/Elliptic -> final_data/Elliptic

Обработка: Amazon Reviews
  Скопирована папка: data/Amazon -> final_data/Amazon

Обработка: Yelp Reviews
  Скопирована папка: data/YelpChi -> final_data/YelpChi

Обработка: Click Fraud
  Скопирован: data/TalkingData/train_sample.csv -> final_data/click_fraud_sample.csv

Обработка: Social Honeypot
    Сохранён Social Honeypot: (41499, 7) -> final_data/social_honeypot.csv

Обработка: MGTAB
  Скопирована папка: data/MGTAB -> final_data/MGTAB

Обработка: Cresci-2017
    Загружено genuine: 3474 за

In [1]:
import pandas as pd
import os

DATA_ROOT = "final_data"

datasets = {
    "Credit Card Fraud": "creditcard.csv",
    "IEEE-CIS": "ieee_train_transaction.csv",
    "PaySim": "paysim.csv",
    "AIZBank": "aizbank.csv",
    "Elliptic (Bitcoin)": "Elliptic/elliptic_txs_features.csv",
    "Amazon Reviews": "Amazon/Amazon_full.csv",
    "Yelp Reviews": "YelpChi/YelpChi_full.csv",
    "Click Fraud": "click_fraud_sample.csv",
    "Social Honeypot": "social_honeypot.csv",
    "Cresci-2017": "cresci_merged.csv",
    "Midterm-2018": "midterm_merged.csv",
    "MGTAB": "MGTAB/features.csv",
}

for name, path in datasets.items():
    full_path = os.path.join(DATA_ROOT, path)
    if not os.path.exists(full_path):
        print(f"{name}: файл не найден")
        continue
    df_sample = pd.read_csv(full_path, nrows=5)
    object_cols = df_sample.select_dtypes(include=['object']).columns.tolist()
    print(f"{name}: object columns = {object_cols}")

Credit Card Fraud: object columns = []
IEEE-CIS: object columns = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9']
PaySim: object columns = ['type', 'nameOrig', 'nameDest']
AIZBank: object columns = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
Elliptic (Bitcoin): object columns = []
Amazon Reviews: object columns = []
Yelp Reviews: object columns = []
Click Fraud: object columns = ['click_time']
Social Honeypot: object columns = []
Cresci-2017: object columns = []
Midterm-2018: object columns = ['probe_timestamp', 'screen_name', 'name', 'description', 'user_created_at', 'url', 'lang']
MGTAB: object columns = []
